# Example: Denoising - Part 1/1

- Author: Dr. Daning Huang
- Date: 05/08/2026
- Updated: 05/08/2026


## Introduction

This notebook follows the denoising workflow currently implemented in `scripts/denoise/noise_proc.py` and `scripts/denoise/noise_train.py`. It covers the standard `DyMAD` example path:

- Data generation
- Model training
- Model prediction

and adds a denoising-focused comparison layer:

- Noisy versus denoised signal plots
- Side-by-side comparison of Savitzky-Golay, Gaussian kernel smoothing, and compact polynomial kernel smoothing
- Quantitative RMSE metrics for preprocessing quality and downstream prediction quality


This notebook deliberately reuses the existing denoise assets instead of copying them into a second workflow:

- `scripts/denoise/noise_data.yaml` for data generation
- `scripts/denoise/noise_wf.yaml` for Savitzky-Golay preprocessing plus weak-form training
- `scripts/denoise/noise_cpp.yaml` for compact polynomial kernel smoothing plus weak-form training

Gaussian kernel smoothing is included in the preprocessing comparison because it is present in `noise_proc.py`, but the repository does not currently include a matching end-to-end training config for it. The notebook keeps that gap explicit.


In [ ]:
import os
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt
import numpy as np
import torch


def find_repo_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "scripts" / "denoise" / "noise_train.py").exists():
            return candidate
    raise FileNotFoundError("Could not locate repo root from the current working directory.")


REPO_ROOT = find_repo_root(Path.cwd())
SCRIPT_DIR = REPO_ROOT / "scripts" / "denoise"

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

os.chdir(SCRIPT_DIR)
print(f"Working directory: {Path.cwd()}")


## Data Generation

We use the same synthetic LTI system as `noise_train.py`. The training dataset is generated from clean trajectories, then observation noise is added before saving the noisy observations to `data/lti_denoise.npz`, which is the file consumed by the training YAML configs.


In [ ]:
from dymad.io import DataInterface, load_model
from dymad.models import LTI
from dymad.training import StackedTrainer
from dymad.utils import TrajectorySampler, plot_summary, plot_trajectory

B = 128
N = 501
t_grid = np.linspace(0, 5, N)

A = np.array([[0.0, 1.0], [-1.0, -0.1]])


def f(t, x, u):
    return (x @ A.T) + u


def g(t, x, u):
    return x


config_chr = {
    "control": {
        "kind": "chirp",
        "params": {
            "t1": 4.0,
            "freq_range": (0.5, 2.0),
            "amp_range": (0.5, 1.0),
            "phase_range": (0.0, 360.0),
        },
    }
}

config_gau = {
    "control": {
        "kind": "gaussian",
        "params": {"mean": 0.5, "std": 1.0, "t1": 4.0, "dt": 0.2, "mode": "zoh"},
    }
}

noise_options = [
    {
        "kind": "gaussian",
        "params": {"mean": 0.0, "std": 0.1},
    },
    {
        "kind": "laplace",
        "params": {"loc": [0.0, 0.0], "scale": [0.1, 0.1]},
    },
    {
        "kind": "student_t",
        "params": {"df": [5.0, 5.0], "loc": [0.0, 0.0], "scale": [0.1, 0.1]},
    },
    {
        "kind": "uniform",
        "params": {"bounds": [[-0.1, 0.1], [-0.1, 0.1]]},
    },
]

noise_kind = 3
config_noise = noise_options[noise_kind]
DATA_PATH = Path("data") / "lti_denoise.npz"


In [ ]:
os.makedirs(DATA_PATH.parent, exist_ok=True)

sampler = TrajectorySampler(
    f,
    g,
    config="noise_data.yaml",
    config_mod={**config_chr, "noise": config_noise},
)
ts_train, x_train_truth, u_train, x_train_noisy = sampler.sample(t_grid, batch=B)
np.savez_compressed(DATA_PATH, t=ts_train, x=x_train_noisy, u=u_train)

print(f"Saved noisy training data to {DATA_PATH.resolve()}")
print(f"Training data shape: {x_train_noisy.shape}")


The saved file contains the noisy observations because that is what the denoise training configs expect. We keep the clean trajectories returned by the sampler in memory so that we can measure how well each denoiser recovers the underlying signal.


In [ ]:
fig, ax = plt.subplots(nrows=2, sharex=True, figsize=(9, 4))
for i in range(2):
    ax[i].plot(ts_train[0], x_train_truth[0, :, i], "k-", label="Truth")
    ax[i].plot(ts_train[0], x_train_noisy[0, :, i], "r:", label="Noisy")
    ax[i].set_ylabel(f"x{i + 1}")
ax[-1].set_xlabel("t")
ax[0].legend()
fig.suptitle("Example training trajectory before denoising")
fig.tight_layout()


## Denoising Algorithms

Savitzky-Golay filtering smooths a signal by sliding a small polynomial fit across the data and evaluating the fitted polynomial at the center of each window. In plain language, it removes noise while trying to preserve local shape, such as peaks and curvature.

Kernel smoothing replaces each sample with a weighted local average of nearby samples. Points close to the current time get larger weights, and distant points get smaller weights. Here we compare two kernels already used by the workflow: Gaussian weights and compact polynomial weights.


The preprocessing comparison below mirrors `noise_proc.py`. All three denoisers are applied to the same noisy trajectories, but only two of them currently have matching training configs in the repository:

- Savitzky-Golay: full preprocessing and training coverage via `noise_wf.yaml`
- Compact polynomial kernel smoothing: full preprocessing and training coverage via `noise_cpp.yaml`
- Gaussian kernel smoothing: preprocessing comparison only, no matching training YAML at the moment


In [ ]:
preprocess_cases = [
    (
        "Savitzky-Golay",
        {
            "type": "denoise",
            "method": "savgol",
            "window_length": 15,
            "polyorder": 5,
        },
    ),
    (
        "Gaussian kernel",
        {
            "type": "denoise",
            "method": "kernel_smoothing",
            "kernel": "gaussian",
            "anchor_count": 64,
            "bandwidth_multiplier": 2.0,
        },
    ),
    (
        "Compact polynomial kernel",
        {
            "type": "denoise",
            "method": "kernel_smoothing",
            "kernel": "compact_polynomial",
            "anchor_count": 64,
            "bandwidth_multiplier": 2.0,
            "degree": 4.0,
        },
    ),
]


def apply_preprocess(config, X):
    interface = DataInterface(
        config_path="noise_wf.yaml",
        config_mod={"transform_x": [config]},
    )
    return interface.encode(X)


def rmse(a, b):
    a = np.asarray(a)
    b = np.asarray(b)
    return float(np.sqrt(np.mean((a - b) ** 2)))


denoised_train = {name: apply_preprocess(config, x_train_noisy) for name, config in preprocess_cases}
signal_rmse = {"Noisy observation": rmse(x_train_noisy, x_train_truth)}
signal_rmse.update({name: rmse(value, x_train_truth) for name, value in denoised_train.items()})

for name, value in signal_rmse.items():
    print(f"{name:28s} RMSE = {value:.6f}")


In [ ]:
fig, ax = plt.subplots(
    nrows=2,
    ncols=len(preprocess_cases),
    sharex=True,
    sharey="row",
    figsize=(14, 5),
)

for j, (name, _) in enumerate(preprocess_cases):
    Z = denoised_train[name]
    for i in range(2):
        ax[i, j].plot(ts_train[0], x_train_truth[0, :, i], "k-", label="Truth")
        ax[i, j].plot(ts_train[0], x_train_noisy[0, :, i], "r:", label="Noisy")
        ax[i, j].plot(ts_train[0], Z[0, :, i], "b-", label="Denoised")
        ax[i, j].set_ylabel(f"x{i + 1}")
    ax[0, j].set_title(name)

for j in range(len(preprocess_cases)):
    ax[-1, j].set_xlabel("t")

ax[0, 0].legend(loc="upper right")
fig.suptitle("Noisy versus denoised signals on one training trajectory")
fig.tight_layout()

fig, ax = plt.subplots(figsize=(8, 3))
ax.bar(
    signal_rmse.keys(),
    signal_rmse.values(),
    color=["tab:red", "tab:blue", "tab:orange", "tab:green"],
)
ax.set_ylabel("RMSE to clean signal")
ax.set_title("Preprocessing quality across denoisers")
ax.tick_params(axis="x", rotation=15)
fig.tight_layout()


## Model Training

The full training flow comes directly from `noise_train.py` and reuses the two existing training configs. Both cases train the same `LTI` model with the `StackedTrainer`; the only difference is the preprocessing phase defined in the YAML file.


In [ ]:
training_cases = [
    {
        "label": "Savitzky-Golay + weak form",
        "config": "noise_wf.yaml",
        "checkpoint": "lti_denoise_wf.pt",
        "summary": "lti_denoise_wf",
    },
    {
        "label": "Compact polynomial kernel + weak form",
        "config": "noise_cpp.yaml",
        "checkpoint": "lti_denoise_cpp.pt",
        "summary": "lti_denoise_cpp",
    },
]

for case in training_cases:
    trainer = StackedTrainer(case["config"], LTI)
    trainer.train()


After training, the saved summaries can be plotted in the same way as the script workflow. This is a convenient check that both denoisers converge before we compare prediction accuracy on a new trajectory.


In [ ]:
plot_summary(
    [case["summary"] for case in training_cases],
    labels=[case["label"] for case in training_cases],
    ifclose=False,
);


## Model Prediction

We now generate a held-out trajectory with a different control input distribution, load the trained checkpoints, and compare how the two trained denoise pipelines predict the clean dynamics from noisy observations.

Gaussian kernel smoothing is not included in this prediction stage because the repository currently does not provide a matching training config or checkpoint workflow for it.


In [ ]:
sampler = TrajectorySampler(
    f,
    g,
    config="noise_data.yaml",
    config_mod={**config_gau, "noise": config_noise},
)

ts_test, x_test_truth, u_test, x_test_noisy = sampler.sample(t_grid, batch=1)
t_test = ts_test[0]
x_truth = x_test_truth[0]
u_test = u_test[0]
y_noisy = x_test_noisy[0]

prediction_series = [x_truth, y_noisy]
prediction_labels = ["Truth", "Noisy observation"]
prediction_rmse = {"Noisy observation": rmse(y_noisy, x_truth)}

for case in training_cases:
    _, predict = load_model(LTI, case["checkpoint"])
    with torch.no_grad():
        pred = np.asarray(predict(y_noisy, t_test, u=u_test))
    prediction_series.append(pred)
    prediction_labels.append(case["label"])
    prediction_rmse[case["label"]] = rmse(pred, x_truth)

for name, value in prediction_rmse.items():
    print(f"{name:36s} RMSE = {value:.6f}")


In [ ]:
plot_trajectory(
    np.array(prediction_series),
    t_test,
    "lti_denoise_prediction",
    us=u_test,
    labels=prediction_labels,
    ifclose=False,
)

fig, ax = plt.subplots(figsize=(8, 3))
ax.bar(
    prediction_rmse.keys(),
    prediction_rmse.values(),
    color=["tab:red", "tab:blue", "tab:green"],
)
ax.set_ylabel("RMSE to clean signal")
ax.set_title("Downstream prediction accuracy")
ax.tick_params(axis="x", rotation=15)
fig.tight_layout()


In this example, Savitzky-Golay, Gaussian kernel smoothing, and compact polynomial kernel smoothing can all be compared at the signal-processing level, while only the Savitzky-Golay and compact polynomial variants currently extend through the full training and prediction workflow. That split is a property of the repository assets today, not a claim about the methods themselves.
